# spatialGlue Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using spatialGlue on simulated dataset.

## Loading

In [ ]:
import SpatialGlue
import omicverse as ov
import pandas as pd
import scanpy as sc
import torch
import os


In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
from SpatialGlue.preprocess import fix_seed
random_seed = 2022
fix_seed(random_seed)

## SpaitalGlue pipeline

In [ ]:
import scanpy as sc
import os
from SpatialGlue.preprocess import clr_normalize_each_cell, pca, lsi,construct_neighbor_graph
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue

# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA and ATAC datasets
    adata_omics1 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_omics2 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    adata_omics2.obsm['spatial'] = adata_omics1.obsm['spatial']
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    # Preprocess RNA data
    sc.pp.filter_genes(adata_omics1, min_cells=10)
    #sc.pp.filter_cells(adata_omics1, min_genes=200)
    #sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
    #sc.pp.normalize_total(adata_omics1, target_sum=1e4)
    #sc.pp.log1p(adata_omics1)
    sc.pp.scale(adata_omics1)
    adata_omics1.obsm['feat'] = pca(adata_omics1, n_comps=50)

    # Preprocess ATAC data
    sc.pp.filter_genes(adata_omics2, min_cells=1)
    lsi(adata_omics2, use_highly_variable=False, n_components=51)
    adata_omics2.obsm['feat'] = adata_omics2.obsm['X_lsi'].copy()

    # Construct neighbor graph
    data_type = 'Spatial-epigenome-transcriptome'
    data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)

    # Define and train the model
    model = Train_SpatialGlue(data, datatype=data_type, device=device)
    output = model.train()

    # Copy the results to the RNA dataset
    adata = adata_omics1.copy()
    adata.obsm['emb_latent_omics1'] = output['emb_latent_omics1']
    adata.obsm['emb_latent_omics2'] = output['emb_latent_omics2']
    adata.obsm['SpatialGlue'] = output['SpatialGlue']
    adata.obsm['alpha'] = output['alpha']
    adata.obsm['alpha_omics1'] = output['alpha_omics1']
    adata.obsm['alpha_omics2'] = output['alpha_omics2']

    # Perform clustering
    ov.pp.neighbors(adata, n_neighbors=15, n_pcs=adata.obsm['SpatialGlue'].shape[1],
                    use_rep='SpatialGlue')
    ov.utils.cluster(adata, use_rep='SpatialGlue', method='leiden', resolution=0.6)

    # Plot the spatial clustering results
    sc.pl.spatial(adata, color=['cell_type', 'leiden'], spot_size=0.12, wspace=0.4)

    # Save the processed dataset
    adata.write_h5ad(f'{output_dir}/Simulated_Dataset_{i}/spatialglue_multiomics.h5ad', compression='gzip')

In [ ]:
!pip list